# Difference Between R-squared and Adjusted R-squared

## R-Squared
- Measures the proportion of the variance in the depndent variable explained by the indeendent variables in the model.
- Ranges from 0 to 1, where 0 indicates the model does not explain any variability, and one indicates it explain all the variability.
- Higher R-Squared values suggest better fit, not neccesarily that the model is a good predictor.

## Adjusted R-Squared
- Addresses a limitation of R-Squared, especially in multiple regression.
- R-Squared tends to increase as more variables are added to the model (even if they don't improve the model significantly), **adjusted r-squared** penalizes the addition of unnecessary variables.
- It considers the number of predictors in the model and adjusts R-Squared accordingly. The adjustment helps to avoid overfitting.

## Residual Sum of Squares

**Residual** for a point in the data is the difference between the actual value and the value predicted by our linear regression model.

$$
Residual = actual - predicted = y-y
$$

Using the residual values, we can determine the sum of squares of the residuals also known as **Residual sum of squares** or **RSS**

$$
RSS = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

The lower the value of **RSS**, the better the model predictions. The falw is that the value depends on the scale of the target variable.

## R-Squared Statistic

**R-Squared statistic** or **coefficient of determination** is a scale invariant statistic that gives the proportion of variation in target variable explained by the linear regression model.

In order to determine the proportion of target variation explained by the model, we need to first determine the following...

### Total Sum of Squares

Total variation in target variable is the sum of squares of the difference between the actual values and their mean.

$$
TSS = \sum_{i=1}^{n} (y_i - \bar{y})^2
$$

We can see that it is very similar to the variance of **Y**. While the variance is the average of the squared sums of difference between actual values and data points, TSS is the total of the squared sums.

Now that we know this, how do we determine the proportion of this variation explained by our model? We go back to **RSS**.

### Residual Sum of Squares 

If we focus on a single residual distance of an actual point from the regression line, we can say that it is the distance that is **not captured by the regression line**, therefore **RSS** gives us the variation in the target variable that is not explained by our model.

### Calculate R-Squared

Therefore R-Squared is equal to...

$$
R^2 = \frac{\text{Explained Variation (TSS - RSS)}}{\text{Total Variation (TSS)}} = 1 - \frac{\text{Unexplained Variation (RSS)}}{\text{Total Variation (TSS)}}
$$


## Adjusted R-Squared Statistics

The adjusted R-Squared takes into account the number of independent variables used for predicting the target variable.

$$
\text{Adjusted } R^2 = \left\{1 - \frac{(1 - R^2)(n - 1)}{(n - k - 1)}\right\}
$$

Where:
- **n** represents the number of data points in our dataset
- **k** represents the number of independent variables, and
- **R** represents the R-squared values determined by the model.

# Python for Data Analysis
    
    AUTHOR: Dr. Wes McKinney 

### Chapter 12: Introduction to Modeling Libraries in Python
### **12.4 Introduction to scikit-learn**

In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train.head(4)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S


Libraries like statsmodels and scikit-learn cannot be fed missing data, so we need to check the columns.

In [3]:
train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

To use the variable Age, as a predictor, we will do a simple imputation of missing values using the median.

In [6]:
impute_value = train['Age'].median()
train['Age'] = train['Age'].fillna(impute_value)
test['Age'] = test['Age'].fillna(impute_value)

We add a column "IsFemale" as an encoded version of the "Sex" column.

In [7]:
train['IsFemale'] = (train['Sex'] == 'female').astype(int)
test['IsFemale'] = (test['Sex'] == 'female').astype(int)

Decide the model variables and create NumPy arrays for the features and target variable.

In [8]:
predictors = ['Pclass', 'IsFemale', 'Age']

X_train = train[predictors]
X_test = test[predictors]
y_train = train['Survived']
y_test = pd.read_csv('gendermodel.csv')['Survived']

In [9]:
y_train[:5]

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

Using the `LogisticRegression` model from `scikit-learn` we create a model instance and fit the model to the training data and form predictions for the test dataset.

In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
y_predict = model.predict(X_test)
y_predict[:10]

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0])

In [11]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

print('Confusion matrix:')
print(confusion_matrix(y_test, y_predict))
print('Precision = TP / (TP + FP) = 144 / (144 + 16) = ', precision_score(y_test, y_predict))
print('Recall = TP / (TP + FN) = 144 / (144 + 8)', recall_score(y_test, y_predict)) 

Confusion matrix:
[[250  16]
 [  8 144]]
Precision = TP / (TP + FP) = 144 / (144 + 16) =  0.9
Recall = TP / (TP + FN) = 144 / (144 + 8) 0.9473684210526315


## Using pipeline

In [12]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train[:5]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer

train['IsFemale'] = (train['Sex'] == 'female').astype(int)
test['IsFemale'] = (test['Sex'] == 'female').astype(int)

predictors = ['Pclass', 'IsFemale', 'Age']

X_train = train[predictors]
X_test = test[predictors]
y_train = train['Survived']

model = make_pipeline(SimpleImputer(strategy='median'), LogisticRegression())
model.fit(X_train, y_train)
y_predict = model.predict(X_test)

print('Confusion matrix:')
print(confusion_matrix(y_test, y_predict))
print('Precision = TP / (TP + FP) = 144 / (144 + 16) = ', precision_score(y_test, y_predict))
print('Recall = TP / (TP + FN) = 144 / (144 + 8)', recall_score(y_test, y_predict)) 

Confusion matrix:
[[250  16]
 [  8 144]]
Precision = TP / (TP + FP) = 144 / (144 + 16) =  0.9
Recall = TP / (TP + FN) = 144 / (144 + 8) 0.9473684210526315
